# 🏆 Agent-Based Financial QA Challenge (UTN FRSR)
## Track Open-Weight en Google Colab (GPU T4 · Microservicio Caja Negra + Cloudflare Tunnel)

Este notebook está dedicado al **Track Open-Weight (100% on-premise en GPU T4 de Google Colab, sin APIs externas)**:
1. Clona el Starter Kit e instala las dependencias base + `cloudflared`.
2. Instala `llama-cpp-python` con soporte CUDA (`cu122`) y registra las librerías compartidas de CUDA 12 requeridas por Colab T4.
3. Levanta el servidor FastAPI (`src/server.py`) en un hilo secundario dentro del mismo kernel (para compartir la VRAM de la GPU T4 con tu modelo GGUF) y expone la URL pública mediante **Cloudflare Tunnel (`cloudflared`)**.

In [ ]:
#@title 1. Clonar Starter Kit, instalar dependencias base y descargar cloudflared { display-mode: "form" }
import os
from pathlib import Path

if not Path("src/schemas.py").exists():
    if not Path("/content/aieng-utn-rag-challenge").exists():
        !git clone -q https://github.com/chebaperez/aieng-utn-rag-challenge.git /content/aieng-utn-rag-challenge
    %cd /content/aieng-utn-rag-challenge
else:
    print("ℹ️ Usando checkout local existente en", Path.cwd())

!pip install -q -r requirements.txt

if not os.path.exists("/usr/local/bin/cloudflared"):
    print("⏳ Descargando binario de Cloudflare Tunnel (cloudflared)...")
    !wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
    !chmod +x /usr/local/bin/cloudflared

print("✅ Entorno base listo.")


In [ ]:
#@title 2. Instalar llama-cpp-python (CUDA) y registrar librerías CUDA 12 en GPU T4 { display-mode: "form" }
import ctypes
import glob
import subprocess
from pathlib import Path

try:
    import llama_cpp  # noqa: F401
    print("✅ llama-cpp-python ya instalado.")
except ImportError:
    print("⏳ Instalando llama-cpp-python con aceleración CUDA para GPU T4...")
    !CMAKE_ARGS="-DGGML_CUDA=on" pip install llama-cpp-python --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu122

!pip install -q huggingface_hub

# Registrar libcudart.so.12 y libcublas.so.12 en ldconfig y precargar con RTLD_GLOBAL para que libllama.so las resuelva en Colab
cudart_hits = sorted(glob.glob("/usr/local/**/libcudart.so.12*", recursive=True))
if not cudart_hits:
    print("⏳ Instalando librerías runtime de CUDA 12 (nvidia-cuda-runtime-cu12, nvidia-cublas-cu12)...")
    !pip install -q nvidia-cuda-runtime-cu12 nvidia-cublas-cu12
    cudart_hits = sorted(glob.glob("/usr/local/**/libcudart.so.12*", recursive=True))

cublas_hits = sorted(glob.glob("/usr/local/**/libcublas*.so.12*", recursive=True))
lib_dirs = sorted({str(Path(p).parent) for p in cudart_hits} | {str(Path(p).parent) for p in cublas_hits})
if lib_dirs:
    Path("/etc/ld.so.conf.d/colab_cuda12.conf").write_text("\n".join(lib_dirs) + "\n")
    subprocess.run(["ldconfig"], check=False)
    for lib_dir in lib_dirs:
        for name in ("libcudart.so.12", "libcublasLt.so.12", "libcublas.so.12"):
            candidate = Path(lib_dir) / name
            if candidate.exists():
                ctypes.CDLL(str(candidate), mode=ctypes.RTLD_GLOBAL)

print("✅ CUDA 12 libs listas desde:", lib_dirs)
print("💡 Recomendación T4: cargar GGUF en archivo único (ej. bartowski/Qwen2.5-7B-Instruct-GGUF / Qwen2.5-7B-Instruct-Q4_K_M.gguf) con n_gpu_layers=-1, n_ctx=16384 y temperature=0.0.")


In [ ]:
#@title 3. Levantar servidor FastAPI (en hilo), probar evaluación local y abrir Cloudflare Tunnel { display-mode: "form" }
import socket
import sys
import threading
import time
from pathlib import Path
import httpx
import uvicorn

if str(Path.cwd()) not in sys.path:
    sys.path.insert(0, str(Path.cwd()))

from scripts.start_tunnel import start_tunnel, stop_tunnel
from src.server import app

def _port_in_use(port: int) -> bool:
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
        return s.connect_ex(("127.0.0.1", port)) == 0

if not _port_in_use(8000):
    # Levantamos uvicorn en un thread daemon para compartir la VRAM con el modelo GGUF montado en este kernel
    config = uvicorn.Config(app, host="0.0.0.0", port=8000, log_level="warning")
    uvicorn_server = uvicorn.Server(config)
    server_thread = threading.Thread(target=uvicorn_server.run, daemon=True)
    server_thread.start()
else:
    print("ℹ️ FastAPI ya estaba escuchando en :8000 (si editaste src/server.py, reiniciá el runtime para recargar cambios).")

# Esperar hasta 20s a que /health responda HTTP 200
health_payload = None
for _ in range(40):
    try:
        r = httpx.get("http://127.0.0.1:8000/health", timeout=2.0)
        if r.status_code == 200:
            health_payload = r.json()
            break
    except Exception:
        pass
    time.sleep(0.5)

if health_payload is None:
    raise RuntimeError("El servidor FastAPI no respondió en http://127.0.0.1:8000/health luego de 20s.")
print("✅ Healthcheck local:", health_payload)

# Prueba rápida contra 5 casos de data/dev_set.json
!python scripts/run_local_eval.py --limit 5

# Cerrar túnel previo si se re-ejecuta la celda
if "tunnel_proc" in globals() and tunnel_proc is not None and tunnel_proc.poll() is None:
    stop_tunnel(tunnel_proc)

# Abrir túnel público gratuito de Cloudflare (reutiliza scripts/start_tunnel.py)
tunnel_proc, public_url = start_tunnel(host="127.0.0.1", port=8000)

print("\n======================================================")
print("🌐 URL PÚBLICA PARA EL EVALUADOR REMOTO (Track Open-Weight):")
print(public_url)
print("======================================================\n")
